In [1]:
# =============================================================================
# NOTEBOOK: 05_dashboard_and_figures.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# NOTEBOOK 05 — Paper Figures + Consolidated HTML Dashboard
#   Consumes every prior artifact and produces:
#     (1) AS-IS / TO-BE swimlane process figure (static PNG for the paper),
#         with per-lane activity counts and monthly human-effort labels — the
#         visual form of the author's Image-3 reduction argument.
#     (2) Sensitivity heatmap (ROI vs. cases_per_month × hourly_wage).
#     (3) Five-axis validation summary chart.
#     (4) A single self-contained HTML dashboard tying results together, with
#         the TRUE cumulative pipeline cost read from cost_ledger.json.
#
# Figures are static (journal Figure use), per the agreed scope. No interactive
# widgets. All example content and code are in English for journal submission.
#
# Inputs (from prior notebooks):
#   artifacts/inference/process_graph.json
#   artifacts/inference/agent2_time.json
#   artifacts/inference/agent3_roi.json
#   artifacts/inference/five_axis_report.json
#   artifacts/cost_ledger.json           (optional; measured pipeline cost)
# Outputs:
#   artifacts/figures/fig_swimlane_asis_tobe.png
#   artifacts/figures/fig_sensitivity_heatmap.png
#   artifacts/figures/fig_five_axis.png
#   artifacts/dashboard.html
# =============================================================================


# %%
# =============================================================================
# Cell 1. Paths + load all artifacts (no API needed in this notebook)
# =============================================================================
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # headless backend (no display needed)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
FIG       = ARTIFACTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


def load(name):
    p = INFER / name
    if not p.exists():
        raise FileNotFoundError(f"[ERROR] {name} missing. Run its notebook first.")
    return json.loads(p.read_text(encoding="utf-8"))


G   = load("process_graph.json")
A2  = load("agent2_time.json")
A3  = load("agent3_roi.json")
AX5 = load("five_axis_report.json")

LEDGER_PATH = ARTIFACTS / "cost_ledger.json"
ledger = json.loads(LEDGER_PATH.read_text(encoding="utf-8")) \
    if LEDGER_PATH.exists() else {}

print(f"[INFO] Loaded process graph ({len(G['as_is']['nodes'])} nodes), "
      f"Agent-2, Agent-3, five-axis report.")
print(f"[INFO] Cost ledger stages: {list(ledger)}")


# %%
# =============================================================================
# Cell 2. Shared constants for figures (grade colors, lane helpers)
# =============================================================================
PARTIAL_RETAIN = G["params"]["partial_retain"]
AI_LANES = {"system", "gpt", "ai", "ai agent"}
AI_LANE_NAME = "AI Agent"

GRADE_COLOR = {
    "full":    "#cfe8cf",   # green  — fully automatable
    "partial": "#fff2cc",   # yellow — AI draft + human review
    "manual":  "#f8cbcb",   # red    — stays manual
    "[MISSING]": "#dddddd",
}

def is_ai(lane: str) -> bool:
    l = (lane or "").lower()
    return l in AI_LANES or "gpt" in l

def human_factor(grade: str) -> float:
    return {"full": 0.0, "partial": PARTIAL_RETAIN, "manual": 1.0}.get(grade, 1.0)

# Per-node estimated monthly minutes (from Agent 2), keyed by id.
est_by_id = {e["id"]: e for e in A2["estimates"]}
def node_monthly_min(nid: str) -> float:
    e = est_by_id.get(nid, {})
    return float(e.get("monthly_minutes", 0.0) or 0.0)


# %%
# =============================================================================
# Cell 3. FIGURE 1 — AS-IS / TO-BE swimlane with counts + effort labels
#
# AS-IS: every node in its actor lane (AI-performed nodes grouped in one AI lane).
# TO-BE: 'full' human nodes vanish from human lanes (moved to AI lane); 'partial'
#        nodes remain as a single human-review box; 'manual' nodes unchanged.
# Each lane header shows: activity count and monthly human minutes, so the
# figure visually carries the reduction argument.
# =============================================================================
def lanes_asis(nodes):
    lanes = {}
    for n in nodes:
        lane = AI_LANE_NAME if is_ai(n["lane"]) else n["lane"]
        lanes.setdefault(lane, []).append(n)
    return lanes

def lanes_tobe(nodes):
    lanes = {}
    for n in nodes:
        g = n["grade"]
        if is_ai(n["lane"]) or g == "full":
            lanes.setdefault(AI_LANE_NAME, []).append(n)   # AI-performed in TO-BE
        else:
            lanes.setdefault(n["lane"], []).append(n)       # partial(review)/manual
    return lanes

def human_minutes_in_lane(items, tobe=False):
    tot = 0.0
    for n in items:
        if is_ai(n["lane"]):
            continue
        m = node_monthly_min(n["id"])
        tot += m * (human_factor(n["grade"]) if tobe else 1.0)
    return tot

def draw_swimlane(ax, lanes, title, tobe=False):
    ax.set_title(title, fontsize=12, fontweight="bold", loc="left")
    box_w, box_h, gap_x, row_h, per_row = 3.2, 0.66, 0.28, 0.95, 6
    y = 0.0
    for lane, items in lanes.items():
        # In TO-BE, drop nodes that carry zero human effort AND are AI-lane.
        show = [n for n in items
                if not (tobe and (is_ai(n["lane"]) or n["grade"] == "full")
                        and lane != AI_LANE_NAME)]
        show = items  # keep all; AI lane shows AI work, human lanes show the rest
        count = len([n for n in show
                     if not (is_ai(n["lane"]) or (tobe and n["grade"] == "full"))]) \
            if lane != AI_LANE_NAME else len(show)
        hmin = human_minutes_in_lane(show, tobe=tobe) if lane != AI_LANE_NAME else 0.0
        header = f"{lane}"
        sub = (f"activities: {count}" if lane == AI_LANE_NAME
               else f"activities: {count}  |  {hmin:,.0f} min/mo")
        ax.text(-0.35, y + row_h / 2, header, ha="right", va="center",
                fontsize=8, fontweight="bold")
        ax.text(-0.35, y + row_h / 2 - 0.28, sub, ha="right", va="center",
                fontsize=6, color="#555")

        drawn = 0
        for n in show:
            # In TO-BE human lanes, skip 'full' nodes (moved to AI lane).
            if tobe and lane != AI_LANE_NAME and n["grade"] == "full":
                continue
            col = drawn % per_row
            rr = drawn // per_row
            xx = col * (box_w + gap_x)
            yy = y - rr * row_h
            color = GRADE_COLOR.get(n["grade"], "#dddddd")
            ax.add_patch(FancyBboxPatch((xx, yy), box_w, box_h,
                         boxstyle="round,pad=0.02", linewidth=0.5,
                         edgecolor="#333", facecolor=color))
            label = n["name"][:32] + ("…" if len(n["name"]) > 32 else "")
            suffix = " (rev.)" if (tobe and n["grade"] == "partial"
                                   and lane != AI_LANE_NAME) else ""
            ax.text(xx + box_w / 2, yy + box_h / 2, label + suffix,
                    ha="center", va="center", fontsize=5)
            drawn += 1
        rows_used = max((drawn - 1) // per_row + 1, 1)
        y -= rows_used * row_h + 0.5

    ax.set_xlim(-3.2, per_row * (box_w + gap_x))
    ax.set_ylim(y, 1.4)
    ax.axis("off")

asis = G["as_is"]["nodes"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 15))
draw_swimlane(ax1, lanes_asis(asis),
              "AS-IS  ·  human activities by organizational lane", tobe=False)
draw_swimlane(ax2, lanes_tobe(asis),
              f"TO-BE  ·  full→AI (removed from human lanes), "
              f"partial→human review kept ({int(PARTIAL_RETAIN*100)}% effort)",
              tobe=True)

# Legend for grade colors.
from matplotlib.patches import Patch
legend = [Patch(facecolor=GRADE_COLOR["full"], edgecolor="#333", label="full → AI"),
          Patch(facecolor=GRADE_COLOR["partial"], edgecolor="#333", label="partial → AI draft + human review"),
          Patch(facecolor=GRADE_COLOR["manual"], edgecolor="#333", label="manual (unchanged)")]
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=8, frameon=False)
plt.tight_layout(rect=[0, 0.03, 1, 1])

SWIM_PATH = FIG / "fig_swimlane_asis_tobe.png"
fig.savefig(SWIM_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] Figure 1 (swimlane) -> {rel(SWIM_PATH)}")


# %%
# =============================================================================
# Cell 4. FIGURE 2 — Sensitivity heatmap (ROI vs cases/month × hourly wage)
#
# We fix partial_retain at its base value and pivot the sensitivity grid to a
# 2-D heatmap, the most decision-relevant slice for a manager.
# =============================================================================
sg = pd.DataFrame(A3["sensitivity_grid"])
base_pr = A2["params"]["partial_retain"]
slice_df = sg[sg["partial_retain"] == base_pr]

pivot = slice_df.pivot_table(index="hourly_wage_usd",
                             columns="cases_per_month",
                             values="roi_pct")

fig2, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn",
               origin="lower")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("cases per month")
ax.set_ylabel("hourly wage (USD)")
ax.set_title(f"ROI (%) sensitivity  ·  partial_retain fixed at {base_pr}",
             fontsize=11, fontweight="bold")
# Annotate each cell.
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, f"{v:,.0f}", ha="center", va="center", fontsize=7,
                color="#000")
fig2.colorbar(im, ax=ax, label="ROI (%)")
plt.tight_layout()
HEAT_PATH = FIG / "fig_sensitivity_heatmap.png"
fig2.savefig(HEAT_PATH, dpi=150, bbox_inches="tight")
plt.close(fig2)
print(f"[INFO] Figure 2 (sensitivity heatmap) -> {rel(HEAT_PATH)}")


# %%
# =============================================================================
# Cell 5. FIGURE 3 — Five-axis validation summary (normalized bar chart)
#
# The axes use different units, so we present each axis's headline metric with
# its own scale label rather than forcing a single normalized radar (which would
# imply false commensurability). A simple annotated bar per axis is honest.
# =============================================================================
ax_axes = AX5["axes"]
acc = ax_axes["1_accuracy"]
rel_ = ax_axes["2_reliability"]
eff = ax_axes["3_efficiency"]
tra = ax_axes["4_transparency"]
rob = ax_axes["5_robustness"]

labels = ["Accuracy\n(grade κ)", "Reliability\n(1 − mean CV)",
          "Transparency\n(rationale cov.)", "Transparency\n(grounded)",
          "Robustness\n(clarify rate)"]
values = [
    acc.get("grade_cohen_kappa") or 0.0,
    1 - (rel_.get("mean_cv_minutes") or 0.0),
    (tra.get("grade_rationale_coverage_pct") or 0.0) / 100.0,
    (tra.get("pct_estimates_grounded") or 0.0) / 100.0,
    (rob.get("clarifying_rate_pct") or 0.0) / 100.0,
]
fig3, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(labels, values, color=["#4c72b0", "#55a868", "#c44e52",
                                     "#8172b2", "#ccb974"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("normalized score (0–1)")
ax.set_title("Five-Axis Validation Summary (illustrative case)",
             fontsize=11, fontweight="bold")
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}",
            ha="center", fontsize=8)
ax.text(0.5, -0.22,
        "Note: axes measure different constructs; bars share a 0–1 scale only "
        "for display. Efficiency (a cost:saving ratio) and time-MAPE are "
        "reported separately in text.",
        transform=ax.transAxes, ha="center", fontsize=6, color="#666")
plt.tight_layout()
AXFIG_PATH = FIG / "fig_five_axis.png"
fig3.savefig(AXFIG_PATH, dpi=150, bbox_inches="tight")
plt.close(fig3)
print(f"[INFO] Figure 3 (five-axis) -> {rel(AXFIG_PATH)}")


# %%
# =============================================================================
# Cell 6. Finalize the Efficiency axis with the TRUE measured pipeline cost
#
# Sum the per-stage costs from the ledger. If the ledger is empty (all cached
# with cost stripped), fall back to the observed first-run total and say so.
# =============================================================================
measured_cost = round(sum(v.get("total_usd", 0.0) for v in ledger.values()), 6)
if measured_cost <= 1e-6:
    measured_cost = 0.017   # observed first-run total across nb 01–04
    cost_note = "first-run observed total (cached re-runs report $0)"
else:
    cost_note = "measured from cost_ledger.json"

annual_saving = A3["baseline"]["annual_saving_usd"]
ratio = round(annual_saving / measured_cost, 0) if measured_cost > 1e-6 else None
print(f"[INFO] Measured pipeline cost: ${measured_cost:.4f} ({cost_note})")
print(f"[INFO] One-time analysis cost vs recurring annual saving: "
      f"${measured_cost:.3f} vs ${annual_saving:,.0f}  (~{ratio:,.0f}:1)")


# %%
# =============================================================================
# Cell 7. Build a single self-contained HTML dashboard
#
# Static HTML with embedded PNGs (base64) so the file is portable — a reviewer
# can open it with no server or dependencies.
# =============================================================================
import base64

def b64(path):
    return base64.b64encode(Path(path).read_bytes()).decode("ascii")

base = A3["baseline"]
counts = G.get("counts", {})
exec_summary = A3.get("executive_summary", "")

html = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<title>GenAI ROI Measurement — Illustrative TCB Case</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;
   margin:0;background:#f6f7f9;color:#1a1a1a}}
 .wrap{{max-width:1100px;margin:0 auto;padding:28px}}
 h1{{font-size:20px;margin:0 0 4px}} h2{{font-size:15px;margin:26px 0 10px;
   border-bottom:2px solid #e2e5ea;padding-bottom:6px}}
 .sub{{color:#666;font-size:12px;margin-bottom:18px}}
 .cards{{display:flex;flex-wrap:wrap;gap:12px}}
 .card{{background:#fff;border:1px solid #e2e5ea;border-radius:10px;
   padding:14px 16px;flex:1;min-width:150px}}
 .card .k{{font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.04em}}
 .card .v{{font-size:22px;font-weight:700;margin-top:4px}}
 img{{max-width:100%;border:1px solid #e2e5ea;border-radius:10px;background:#fff}}
 .cav{{background:#fff8e1;border:1px solid #ffe082;border-radius:8px;
   padding:10px 14px;font-size:12px;color:#7a5c00;margin-top:10px}}
 table{{border-collapse:collapse;width:100%;font-size:12px;background:#fff}}
 td,th{{border:1px solid #e2e5ea;padding:6px 10px;text-align:left}}
 th{{background:#f0f2f5}}
 .foot{{color:#999;font-size:11px;margin-top:30px}}
</style></head><body><div class="wrap">
<h1>Automating Interview-Based Generative-AI ROI Measurement</h1>
<div class="sub">Domain-agnostic three-agent pipeline · illustrative case:
 technology-credit-evaluation interview · model tier: weak (gpt-4o-mini)</div>

<h2>Headline results (baseline scenario)</h2>
<div class="cards">
 <div class="card"><div class="k">Human-effort reduction</div>
   <div class="v">{base['effort_reduction_pct']:.1f}%</div></div>
 <div class="card"><div class="k">Saved hours / year</div>
   <div class="v">{base['saved_hours_year']:,.0f}</div></div>
 <div class="card"><div class="k">Annual saving</div>
   <div class="v">${base['annual_saving_usd']:,.0f}</div></div>
 <div class="card"><div class="k">ROI</div>
   <div class="v">{base['roi_pct']:,.0f}%</div></div>
 <div class="card"><div class="k">Payback</div>
   <div class="v">{base['payback_years']:.2f} yr</div></div>
</div>
<div class="cav">{A3.get('provenance_caveat','')}</div>

<h2>Executive summary (LLM interpretation — no new numbers)</h2>
<p style="font-size:13px;line-height:1.55">{exec_summary}</p>

<h2>Figure 1 — AS-IS / TO-BE process (human activities by lane)</h2>
<img src="data:image/png;base64,{b64(SWIM_PATH)}" alt="swimlane">

<h2>Figure 2 — ROI sensitivity (cases/month × hourly wage)</h2>
<img src="data:image/png;base64,{b64(HEAT_PATH)}" alt="heatmap">

<h2>Figure 3 — Five-axis validation summary</h2>
<img src="data:image/png;base64,{b64(AXFIG_PATH)}" alt="five-axis">

<h2>Five-axis validation (headline metrics)</h2>
<table>
 <tr><th>Axis</th><th>Metric</th><th>Value</th></tr>
 <tr><td>1 Accuracy</td><td>grade κ / grade agreement / time MAPE</td>
   <td>{acc.get('grade_cohen_kappa')} / {acc.get('grade_agreement')} /
       {acc.get('time_MAPE_pct')}%</td></tr>
 <tr><td>2 Reliability</td><td>mean CV across {rel_.get('n_rollouts')} rollouts</td>
   <td>{rel_.get('mean_cv_minutes')}</td></tr>
 <tr><td>3 Efficiency</td><td>one-time cost vs annual saving</td>
   <td>${measured_cost:.3f} vs ${annual_saving:,.0f} (~{ratio:,.0f}:1)</td></tr>
 <tr><td>4 Transparency</td><td>rationale coverage / grounded estimates</td>
   <td>{tra.get('grade_rationale_coverage_pct')}% /
       {tra.get('pct_estimates_grounded')}%</td></tr>
 <tr><td>5 Robustness</td><td>clarifying-question rate / [MISSING] flagged</td>
   <td>{rob.get('clarifying_rate_pct')}% / {rob.get('missing_system_flagged')}</td></tr>
</table>

<div class="foot">Adequacy thresholds per axis are intentionally left as future
 standardization work. All monetary figures derive from prior/implied time
 estimates; the interview stated no explicit durations or volumes. Cost source:
 {cost_note}.</div>
</div></body></html>"""

DASH_PATH = ARTIFACTS / "dashboard.html"
DASH_PATH.write_text(html, encoding="utf-8")
print(f"[INFO] Dashboard -> {rel(DASH_PATH)}")
print("[INFO] Open artifacts/dashboard.html in a browser to view everything.")

[INFO] Loaded process graph (46 nodes), Agent-2, Agent-3, five-axis report.
[INFO] Cost ledger stages: ['01_agent1']
[INFO] Figure 1 (swimlane) -> artifacts\figures\fig_swimlane_asis_tobe.png
[INFO] Figure 2 (sensitivity heatmap) -> artifacts\figures\fig_sensitivity_heatmap.png
[INFO] Figure 3 (five-axis) -> artifacts\figures\fig_five_axis.png
[INFO] Measured pipeline cost: $0.0170 (first-run observed total (cached re-runs report $0))
[INFO] One-time analysis cost vs recurring annual saving: $0.017 vs $149,065  (~8,768,529:1)
[INFO] Dashboard -> artifacts\dashboard.html
[INFO] Open artifacts/dashboard.html in a browser to view everything.
